In [59]:
from typing import Optional
from dotenv import load_dotenv
load_dotenv()
import os
import logging as logger
from azure.core.credentials import AzureKeyCredential
from openai import AzureOpenAI, OpenAI
from app.utils.azure_ai_search_retriever import AzureAISearchRetriever, SearchCredential
from app.utils.rag_orchestrator import RAGOrchestrator


In [61]:
print("Loading environment variables...")
key = os.getenv("AZURE_SEARCH_API_KEY")
print(key)


Loading environment variables...
ErzblwvyWyp8l9M6UlfhQ3FbHiIp0PUGHyXbdsBD5uF0ipYeo7dSJQQJ99BCAAAAAAAAAAAAINFRAZML3nra


In [ ]:
def create_search_credential() -> SearchCredential:
    """Creates the appropriate Azure Search credential based on environment variables."""
    api_key = os.getenv("AZURE_SEARCH_API_KEY")
    print(api_key)
    if api_key:
        logger.info("Using Azure Search API Key credential.")
        return AzureKeyCredential(api_key)


def create_openai_client(purpose: str = "embedding") -> Optional[AzureOpenAI | OpenAI]:
    """Creates an AzureOpenAI client instance based on environment variables."""
    if purpose == "embedding":
        endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
        api_key = os.getenv("AZURE_OPENAI_API_KEY")
        api_version = os.getenv("AZURE_OPENAI_API_VERSION")
        log_prefix = "Embedding"
    elif purpose == "chat":
        endpoint = None
        api_version = None
        api_key = os.getenv("AZURE_OPENAI_CHAT_API_KEY", os.getenv("AZURE_OPENAI_API_KEY"))
        log_prefix = "Chat"
    else:
        raise ValueError(f"Invalid purpose for OpenAI client: {purpose}")

    try:
        if endpoint is None or api_version is None:
            client = OpenAI(
                api_key=api_key,
            )
        else:
            client = AzureOpenAI(
                azure_endpoint=endpoint,
                api_key=api_key,
                api_version=api_version,
            )
        # Perform a simple test if needed (e.g., list models), but might add latency
        logger.info(
            f"Azure OpenAI client for {log_prefix} initialized successfully (Endpoint: {endpoint}, API Version: {api_version}).")
        return client
    except Exception as e:
        logger.error(f"Failed to initialize Azure OpenAI {log_prefix} client: {e}", exc_info=True)
        return None  # Return None instead of raising, allow graceful failure if only one part is needed

In [54]:
# --- Azure Search Configuration ---
search_service_name = os.getenv("AZURE_SEARCH_SERVICE_NAME")
index_name = os.getenv("AZURE_SEARCH_INDEX_NAME")
search_dns_suffix = os.getenv("AZURE_SEARCH_DNS_SUFFIX","search.windows.net")

if not search_service_name or not index_name:
    raise ValueError("Missing required environment variables: AZURE_SEARCH_SERVICE_NAME and AZURE_SEARCH_INDEX_NAME")

search_service_endpoint = f"https://{search_service_name}.search.windows.net"
search_credential = create_search_credential()  # Handles API key, SPN, DefaultAzureCredential

# --- Azure OpenAI Configuration ---
# Embedding Client (Optional for Retriever if only doing text search)
openai_embedding_client = create_openai_client(purpose="embedding")
openai_embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")

# Chat Client (Required for Orchestrator)
openai_chat_client = create_openai_client(purpose="chat")
openai_chat_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

if not openai_chat_client or not openai_chat_deployment:
    raise ValueError(
        "Azure OpenAI Chat client configuration (Endpoint, Key, Version, Deployment) is required and incomplete.")

2025-03-30 19:34:17,434 - INFO - Using Azure Search API Key credential.
2025-03-30 19:34:17,435 - INFO - Using Azure Search API Key: ErzblwvyWyp8l9M6UlfhQ3FbHiIp0PUGHyXbdsBD5uF0ipYeo7dSJQQJ99BCAAAAAAAAAAAAINFRAZML3nra
2025-03-30 19:34:17,880 - INFO - Azure OpenAI client for Embedding initialized successfully (Endpoint: None, API Version: None).
2025-03-30 19:34:18,383 - INFO - Azure OpenAI client for Chat initialized successfully (Endpoint: None, API Version: None).


In [62]:
logger.info(f"AZURE_SEARCH_SERVICE_NAME: {search_service_name}")
logger.info(f"AZURE_SEARCH_INDEX_NAME: {index_name}")
print("CHECK AZURE SEARCH DNS FROM ENV", os.getenv("AZURE_SEARCH_DNS_SUFFIX")) 
logger.info(f"AZURE_SEARCH_DNS_SUFFIX: {search_dns_suffix}")
logger.info(f"search_service_endpoint: {search_service_endpoint}")

2025-03-30 19:44:27,432 - INFO - AZURE_SEARCH_SERVICE_NAME: autograder-search
2025-03-30 19:44:27,433 - INFO - AZURE_SEARCH_INDEX_NAME: assignment2lookup2
2025-03-30 19:44:27,436 - INFO - AZURE_SEARCH_DNS_SUFFIX: https://autograder-search
2025-03-30 19:44:27,437 - INFO - search_service_endpoint: https://autograder-search.search.windows.net


CHECK AZURE SEARCH DNS FROM ENV https://autograder-search


In [63]:
# --- Instantiate Retriever ---
retriever = AzureAISearchRetriever(
    search_service_endpoint=search_service_endpoint,
    index_name=index_name,
    search_credential=search_credential,
    openai_client=openai_embedding_client,  # Pass the client instance
    openai_embedding_deployment=openai_embedding_deployment  # Pass the deployment name
    # select_fields=None, # Use default
    # embedding_dimension=1536 # Use default
)
logger.info("AzureAISearchRetriever instantiated.")

# --- Instantiate Orchestrator ---
orchestrator = RAGOrchestrator(
    retriever=retriever,
    chat_client=openai_chat_client,  # Pass the client instance
    chat_deployment=openai_chat_deployment  # Pass the deployment name
    # system_prompt="Your custom system prompt here", # Optional
    # context_template="Source: {source}\nContent: {content}\n---\n", # Optional
)
logger.info("RAGOrchestrator instantiated.")

2025-03-30 19:44:32,846 - INFO - Initializing AzureAISearchRetriever for endpoint: https://autograder-search.search.windows.net, index: assignment2lookup2
2025-03-30 19:44:32,857 - INFO - Search client initialized successfully for index: assignment2lookup2
2025-03-30 19:44:32,859 - INFO - Performing initial connection test query...
2025-03-30 19:44:32,870 - INFO - Request URL: 'https://autograder-search.search.windows.net/indexes('assignment2lookup2')/docs/search.post.search?api-version=REDACTED'
Request method: 'POST'
Request headers:
    'Content-Type': 'application/json'
    'Content-Length': '40'
    'api-key': 'REDACTED'
    'Accept': 'application/json;odata.metadata=none'
    'x-ms-client-request-id': 'ee238461-0dc0-11f0-8178-00090faa0001'
    'User-Agent': 'azsdk-python-search-documents/11.5.2 Python/3.12.3 (Windows-11-10.0.26100-SP0)'
A body is sent with the request
2025-03-30 19:44:34,386 - INFO - Response status: 403
Response headers:
    'Content-Length': '112'
    'Content-

PermissionError: Connection test failed (403 Forbidden). Verify the provided credential has permissions ('Search Index Data Reader' role needed for RBAC) for index 'assignment2lookup2' at 'https://autograder-search.search.windows.net'. RBAC roles may take time to propagate.

In [ ]:
# --- Example Query ---
user_query = "Why do we need to do Business Process Re-engineering as a part of implementing an HIS/EHR? Note: Your answer must be in your own words."
logger.info(f"Answering query: '{user_query}'")

answer, sources = orchestrator.answer_query(
    user_query=user_query,
    top_k_retrieval=3,
    search_type="hybrid",  # Try hybrid search
    use_semantic_ranking=False  # Set to True if configured and desired
)

if answer:
    print("\n--- Answer ---")
    print(answer)
    print("\n--- Sources ---")
    if sources:
        for source in sources:
            print(f"- {source}")
    else:
        print("No sources were cited (or retrieval failed).")
else:
    print("\nFailed to get an answer.")